# Paper Figures: Figure S2 group similarities showing individual rats

This notebook generates publication-ready figures for the Bazzino & Roitman sodium appetite manuscript using data assembled by `src/assemble_all_data.py`.

**Figure S2: Analysis rat-by-rat** — Comparing different groups.

In [ ]:
import pathlib
from pathlib import Path
import sys
# Add src to path for importing local modules
sys.path.insert(0, str(Path("../src").resolve()))
from data_io import load_assembled

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from trompy import save_figure_atomic
from extract_behav_parameters import smooth_array

from figure_config import (
    configure_matplotlib, COLORS, HEATMAP_CMAP_DIV,
    HEATMAP_CMAP_RED, HEATMAP_CMAP_BLUE,
    DATAFOLDER, FIGSFOLDER,
    SAVE_FIGS
)
from figure_plotting import (
    smooth_array, get_heatmap_data_by_rat, get_mean_snips,
    calculate_ylims,
)

# Configure matplotlib
configure_matplotlib()
colors = COLORS  # Use shared color palette
custom_cmap_red = HEATMAP_CMAP_RED  # Use shared colormap
custom_cmap_blue = HEATMAP_CMAP_BLUE  # Use shared colormap

custom_cmap = HEATMAP_CMAP_DIV  # Use shared diverging colormap

SAVE_FIGS = True
SIMBA_METRIC = "median" # other options... include "zscore"

## Load Assembled Data

Load the complete dataset from the pickle file generated by the assembly script.

In [ ]:
assembled_data_path = DATAFOLDER / "assembled"

data = load_assembled(assembled_data_path)

# Extract main components
x_array = data["x_array"]
snips_photo = data["snips_photo"]

if SIMBA_METRIC == "zscore":
    snips_simba = data["snips_simba_zscore"]
    simba_col = "simba_mean_zscore_auc"
elif SIMBA_METRIC == "median":
    snips_simba = data["snips_simba_baseline"]
    simba_col = "simba_median_balance_score"
else:
    raise ValueError(f"Unknown SIMBA_METRIC: {SIMBA_METRIC}")

params = data.get("params", {})
metadata = data.get("metadata", {})

print("Loaded assembled data from data/assembled")
print(f"\nData structure:")
print(f"  - x_array shape: {x_array.shape}")
print(f"  - snips_photo shape: {snips_photo.shape}")
print(f"  - snips_simba shape: {snips_simba.shape}")
print(f"  - x_array columns: {x_array.columns.tolist()}")
print(f"  - Number of trials: {len(x_array)}")

## Figure S2: Heatmaps of individual rats per group

In [ ]:
snips_to_plot = np.array(smooth_array(snips_simba))  # Use smoothed and z-scored data for all subsequent analyses and plotting

# Parameters for visualization - MOVEMENT
# Use dynamic scaling based on actual data ranges (asymmetric, not symmetric)
# This ensures heatmaps use the full range of your data without artificial symmetry
vmin = np.nanpercentile(snips_to_plot, 5)  # 5th percentile for lower bound
vmax = np.nanpercentile(snips_to_plot, 95)  # 95th percentile for upper bound
print(f"Calculated asymmetric vlims: vmin={vmin:.4f}, vmax={vmax:.4f}")

if SIMBA_METRIC == "zscore":
    vlim = (-1, 1)
elif SIMBA_METRIC == "median":
    vlim = (-0.2, 0.2)
else:
    vlim = (vmin, vmax)  # Use calculated asymmetric limits for other metrics

# Calculate dynamic y-limits based on all snips data - MOVEMENT
print("\n" + "="*60)
print("CALCULATING Y-LIMITS FOR TIME SERIES PLOTS")
print("="*60)

# Get all condition/infusion combinations to compute limits
rep_10, rep_45 = get_mean_snips(snips_simba, x_array, "replete")
dep_10, dep_45 = get_mean_snips(snips_simba, x_array, "deplete")

# Calculate y-limits based on all snips with 5% padding
calc_ylims = calculate_ylims([rep_10, rep_45, dep_10, dep_45], pad_percentage=5)

print(f"  Data range (min, max): ({np.nanmin(snips_simba):.4f}, {np.nanmax(snips_simba):.4f})")
print(f"  Calculated y-limits: {calc_ylims}")
print("="*60 + "\n")

# Use calculated limits or override with fixed values if preferred
ylims = calc_ylims
ylims = (0, 8)

In [ ]:
### 2A-D Heatmaps and time series of SIMBA snips by rat by group

def make_line_plot_in_axis(data_to_plot, ax=None, color="black"):
    if ax is None:
        ax = plt.gca()

    ax.plot(data_to_plot.mean(axis=0), color=color)
    ax.fill_between(range(data_to_plot.shape[1]),
                        data_to_plot.mean(axis=0) - (data_to_plot.std(axis=0) / np.sqrt(data_to_plot.shape[0])),
                        data_to_plot.mean(axis=0) + (data_to_plot.std(axis=0) / np.sqrt(data_to_plot.shape[0])), alpha=0.3, color=color)

    ax.set_xticks([])
    ax.set_yticks([])
    sns.despine(ax=ax, left=True, bottom=True)

heatmap_data_rep_10 = get_heatmap_data_by_rat(snips_to_plot, x_array, "replete", "10NaCl")
heatmap_data_rep_45 = get_heatmap_data_by_rat(snips_to_plot, x_array, "replete", "45NaCl")
heatmap_data_dep_10 = get_heatmap_data_by_rat(snips_to_plot, x_array, "deplete", "10NaCl")
heatmap_data_dep_45 = get_heatmap_data_by_rat(snips_to_plot, x_array, "deplete", "45NaCl")

f, ax = plt.subplots(ncols=5, nrows=2, figsize=(7, 3.6),
                     gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.1],
                                  "left": 0.1, "right": 0.95, "wspace": 0.1})

sns.heatmap(heatmap_data_rep_10, ax=ax[0,0], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_rep_45, ax=ax[0,1], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_dep_10, ax=ax[0,2], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_dep_45, ax=ax[0,3], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=True, cbar_ax=ax[0,4])



for axis in ax[0]:
    axis.set_xticks([])
    axis.set_yticks([])
    axis.plot((149, 198), (10.5, 10.5), color="black", lw=2, clip_on=False)  # Add time scale bar (0.5s at 300Hz)
    axis.plot((49, 148), (-0.5, -0.5), color="black", lw=2, alpha=0.5, clip_on=False)  # Add time scale bar (0.5s at 300Hz)

ax[0,0].set_yticks([0.5, 9.5], labels=["1", "10"], fontsize=10, rotation=0)
ax[0,0].set_ylabel("Rats")

make_line_plot_in_axis(heatmap_data_rep_10, ax=ax[1,0], color=colors[0])
make_line_plot_in_axis(heatmap_data_rep_45, ax=ax[1,1], color=colors[1])
make_line_plot_in_axis(heatmap_data_dep_10, ax=ax[1,2], color=colors[2])
make_line_plot_in_axis(heatmap_data_dep_45, ax=ax[1,3], color=colors[3])

for a in ax[1, 1:]:
    a.sharey(ax[1, 0])

ylims = []
for axis in ax[1,:-1]:
    ylims.append(axis.get_ylim())

for axis in ax[1,:-1]:
    axis.fill_betweenx([np.array(ylims).min(), np.array(ylims).max()], 50, 150, color="grey", alpha=0.1, zorder=-20)


ax[1,0].plot([0,0], [0.05, 0.1], color="k", lw=2)  # Add vertical line to indicate time of infusion

ax[1,4].axis("off")  # Hide the empty subplot used for the colorbar

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S2_simba_groups_heatmaps", FIGSFOLDER)

In [ ]:
snips_to_plot = snips_photo

vlim = (-2, 2) 


heatmap_data_rep_10 = get_heatmap_data_by_rat(snips_to_plot, x_array, "replete", "10NaCl")
heatmap_data_rep_45 = get_heatmap_data_by_rat(snips_to_plot, x_array, "replete", "45NaCl")
heatmap_data_dep_10 = get_heatmap_data_by_rat(snips_to_plot, x_array, "deplete", "10NaCl")
heatmap_data_dep_45 = get_heatmap_data_by_rat(snips_to_plot, x_array, "deplete", "45NaCl")

f, ax = plt.subplots(ncols=5, nrows=2, figsize=(7, 3.6),
                     gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.1],
                                  "left": 0.1, "right": 0.95, "wspace": 0.1})

sns.heatmap(heatmap_data_rep_10, ax=ax[0,0], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_rep_45, ax=ax[0,1], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_dep_10, ax=ax[0,2], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=False)
sns.heatmap(heatmap_data_dep_45, ax=ax[0,3], cmap=custom_cmap, vmin=vlim[0], vmax=vlim[1], cbar=True, cbar_ax=ax[0,4])


for axis in ax[0]:
    axis.set_xticks([])
    axis.set_yticks([])
    axis.plot((149, 198), (10.5, 10.5), color="black", lw=2, clip_on=False)  # Add time scale bar (0.5s at 300Hz)
    axis.plot((49, 148), (-0.5, -0.5), color="black", lw=2, alpha=0.5, clip_on=False)  # Add time scale bar (0.5s at 300Hz)

ax[0,0].set_yticks([0.5, 9.5], labels=["1", "10"], fontsize=10, rotation=0)
ax[0,0].set_ylabel("Rats")

make_line_plot_in_axis(heatmap_data_rep_10, ax=ax[1,0], color=colors[0])
make_line_plot_in_axis(heatmap_data_rep_45, ax=ax[1,1], color=colors[1])
make_line_plot_in_axis(heatmap_data_dep_10, ax=ax[1,2], color=colors[2])
make_line_plot_in_axis(heatmap_data_dep_45, ax=ax[1,3], color=colors[3])

for a in ax[1, 1:]:
    a.sharey(ax[1, 0])

ylims = []
for axis in ax[1,:-1]:
    ylims.append(axis.get_ylim())

for axis in ax[1,:-1]:
    axis.fill_betweenx([np.array(ylims).min(), np.array(ylims).max()], 50, 150, color="grey", alpha=0.1, zorder=-20)


ax[1,0].plot([0,0], [0.5, 1.5], color="k", lw=2)  # Add vertical line to indicate time of infusion

ax[1,4].axis("off")  # Hide the empty subplot used for the colorbar


if SAVE_FIGS:
    save_figure_atomic(f, "figure_S3_photo_groups_heatmaps", FIGSFOLDER)